In [8]:
import os
import urllib.request
import pandas as pd

# Create folder
os.makedirs("../data/RSV_F", exist_ok=True)

# Direct link to the RSV F summary CSV (all antibodies in one file)
url = "https://raw.githubusercontent.com/dms-vep/RSV_Long_F_DMS/main/results/summaries/all_antibodies.csv"
out_path = "../data/RSV_F/rsv_all_antibodies.csv"

if not os.path.exists(out_path):
    print("Downloading RSV F data...")
    urllib.request.urlretrieve(url, out_path)
    print("Downloaded")
else:
    print("Already exists")

# Load and inspect
df_rsv = pd.read_csv(out_path)
print("\nShape:", df_rsv.shape)
print("\nColumns:")
print(df_rsv.columns.tolist())
print("\nFirst 5 rows:")
print(df_rsv.head())

# Quick stats
print(f"\nTotal rows: {len(df_rsv)}")
print(f"Unique sites: {df_rsv['site'].nunique() if 'site' in df_rsv.columns else 'check columns'}")

# If 'antibody' column exists, check the panel
if 'antibody' in df_rsv.columns:
    print(f"Antibodies: {df_rsv['antibody'].nunique()}")
    print(df_rsv['antibody'].value_counts())

Downloaded

Shape: (9824, 20)

Columns:
['site', 'wildtype', 'mutant', 'Nirsevimab-IgG escape', 'Nirsevimab-Fab escape', 'Clesrovimab-IgG escape', 'Clesrovimab-Fab escape', 'Palivizumab-IgG escape', 'Palivizumab-Fab escape', 'RSM01-IgG escape', 'RSM01-Fab escape', 'Suptavumab-IgG escape', 'Suptavumab-Fab escape', '1A2-IgG escape', '1A2-Fab escape', '1B6-IgG escape', '1B6-Fab escape', 'cell entry', 'sequential_site', 'region']

First 5 rows:
   site wildtype mutant  Nirsevimab-IgG escape  Nirsevimab-Fab escape  \
0    26        Q      A                0.07960                0.04561   
1    26        Q      C                0.03003                0.11000   
2    26        Q      D                0.10040               -0.08516   
3    26        Q      E                0.03745                0.47580   
4    26        Q      F               -0.36490               -0.53700   

   Clesrovimab-IgG escape  Clesrovimab-Fab escape  Palivizumab-IgG escape  \
0                 0.03111              

In [9]:
# Melt to long format
escape_cols = [c for c in df_rsv.columns if 'escape' in c and c != 'cell entry']

df_long = df_rsv.melt(
    id_vars=['site', 'wildtype', 'mutant', 'sequential_site', 'region', 'cell entry'],
    value_vars=escape_cols,
    var_name='antibody_full',
    value_name='escape_mean'
)

# Extract antibody name and format (IgG/Fab)
df_long['format'] = df_long['antibody_full'].str.extract(r'-(IgG|Fab) escape')
df_long['antibody'] = df_long['antibody_full'].str.replace(r'-(IgG|Fab) escape', '', regex=True)

# Filter to IgG only (more physiological)
df_igg = df_long[df_long['format'] == 'IgG'].copy()

print("Long format shape:", df_igg.shape)
print("Antibodies:", df_igg['antibody'].unique())
print("Rows per antibody:")
print(df_igg.groupby('antibody').size())

# Binary label
df_igg['label'] = (df_igg['escape_mean'] > 0.5).astype(int)
print(f"\nPositive rate @ 0.5: {df_igg['label'].mean():.4f}")
print(f"Positive rate @ top10%: {(df_igg['escape_mean'] > df_igg['escape_mean'].quantile(0.9)).mean():.4f}")
print(f"Positive rate @ top5%: {(df_igg['escape_mean'] > df_igg['escape_mean'].quantile(0.95)).mean():.4f}")

Long format shape: (68768, 10)
Antibodies: <ArrowStringArray>
['Nirsevimab', 'Clesrovimab', 'Palivizumab', 'RSM01', 'Suptavumab', '1A2',
 '1B6']
Length: 7, dtype: str
Rows per antibody:
antibody
1A2            9824
1B6            9824
Clesrovimab    9824
Nirsevimab     9824
Palivizumab    9824
RSM01          9824
Suptavumab     9824
dtype: int64

Positive rate @ 0.5: 0.0335
Positive rate @ top10%: 0.0831
Positive rate @ top5%: 0.0415


In [16]:
import pandas as pd

df_rsv = pd.read_csv("../data/RSV_F/rsv_all_antibodies.csv")
print("Reloaded shape:", df_rsv.shape)

escape_cols = [c for c in df_rsv.columns if 'escape' in c and c != 'cell entry']

df_long = df_rsv.melt(
    id_vars=['site', 'wildtype', 'mutant', 'sequential_site', 'region', 'cell entry'],
    value_vars=escape_cols,
    var_name='antibody_full',
    value_name='escape_mean'
)

df_long['format'] = df_long['antibody_full'].str.extract(r'-(IgG|Fab) escape')
df_long['antibody'] = df_long['antibody_full'].str.replace(r'-(IgG|Fab) escape', '', regex=True)
df_igg = df_long[df_long['format'] == 'IgG'].copy()
df_igg['label'] = (df_igg['escape_mean'] > 0.5).astype(int)

print("Long shape:", df_igg.shape)
print("Antibodies:", df_igg['antibody'].unique())
print("Positive rate:", df_igg['label'].mean())

df_igg.to_csv("../data/RSV_F/rsv_long_igg.csv", index=False)
print("Saved.")

Reloaded shape: (9824, 20)
Long shape: (68768, 11)
Antibodies: <ArrowStringArray>
['Nirsevimab', 'Clesrovimab', 'Palivizumab', 'RSM01', 'Suptavumab', '1A2',
 '1B6']
Length: 7, dtype: str
Positive rate: 0.03348941368078176
Saved.


In [17]:
df_igg = pd.read_csv("../data/RSV_F/rsv_long_igg.csv")
seq_df = df_igg.groupby('sequential_site')['wildtype'].agg(lambda x: x.mode()[0]).sort_index()
rsv_seq = ''.join(seq_df.values)
with open("../data/RSV_F/rsv_sequence.txt", "w") as f:
    f.write(rsv_seq)
print("Saved sequence, length:", len(rsv_seq))

Saved sequence, length: 504


In [1]:
import numpy as np
import torch
import esm
import os

rsv_seq = open("../data/RSV_F/rsv_sequence.txt").read().strip()
print("Sequence length:", len(rsv_seq))

os.makedirs("../data/RSV_F/esm_cache", exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Use 35M instead of 650M
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
model = model.eval().to(device)
batch_converter = alphabet.get_batch_converter()
REPR_LAYER = 12

_, _, tokens = batch_converter([("RSV_F", rsv_seq)])
tokens = tokens.to(device)
with torch.no_grad():
    out = model(tokens, repr_layers=[REPR_LAYER])
rsv_emb = out["representations"][REPR_LAYER][0, 1:len(rsv_seq)+1].cpu().numpy()
print("Embedding shape:", rsv_emb.shape)  # should be (504, 480)

np.save("../data/RSV_F/esm_cache/RSV_F_esm_35M.npy", rsv_emb)
print("Saved.")

# Free GPU memory
del model
if device == "cuda":
    torch.cuda.empty_cache()
print("GPU cache cleared.")

Sequence length: 504
Device: cuda
Embedding shape: (504, 480)
Saved.
GPU cache cleared.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
df_igg = pd.read_csv("../data/RSV_F/rsv_long_igg.csv")
df_igg['seq_idx'] = df_igg['sequential_site'] - 1
df_igg['wt_mut'] = df_igg['wildtype'] + df_igg['mutant']

rsv_emb = np.load("../data/RSV_F/esm_cache/RSV_F_esm_35M.npy")
print("Embedding:", rsv_emb.shape)

enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.float32)
X_cat = enc.fit_transform(df_igg[['wt_mut']]).astype(np.float32)
X_pos = df_igg[['seq_idx']].values.astype(np.float32)
X_g   = rsv_emb[df_igg['seq_idx'].values].astype(np.float32)

print("X_cat:", X_cat.shape)
print("X_pos:", X_pos.shape)
print("X_g:", X_g.shape)

df_igg.to_csv("../data/RSV_F/rsv_igg_features.csv", index=False)
print("Saved.")

Embedding: (504, 480)
X_cat: (68768, 400)
X_pos: (68768, 1)
X_g: (68768, 480)
Saved.


In [7]:
import pandas as pd
import numpy as np

# Reload raw
df_rsv = pd.read_csv("../data/RSV_F/rsv_all_antibodies.csv")
print("Raw shape:", df_rsv.shape)

# Melt
escape_cols = [c for c in df_rsv.columns if 'escape' in c and c != 'cell entry']
df_long = df_rsv.melt(
    id_vars=['site', 'wildtype', 'mutant', 'sequential_site', 'region', 'cell entry'],
    value_vars=escape_cols,
    var_name='antibody_full',
    value_name='escape_mean'
)
df_long['format'] = df_long['antibody_full'].str.extract(r'-(IgG|Fab) escape')
df_long['antibody'] = df_long['antibody_full'].str.replace(r'-(IgG|Fab) escape', '', regex=True)

print("Long shape:", df_long.shape)  # should be ~137,536 (9824 × 14)
print("Format counts:")
print(df_long['format'].value_counts())

# IgG only
df_igg = df_long[df_long['format'] == 'IgG'].copy()
print("\nIgG shape:", len(df_igg))  # should be ~68,768

# Filter to only rows with valid escape_mean
df_igg = df_igg[df_igg['escape_mean'].notna()].copy()
print("After dropna:", len(df_igg))

print("Rows per antibody:")
print(df_igg.groupby('antibody').size())

# Label
df_igg['label'] = (df_igg['escape_mean'] > 0.5).astype(int)
print("\nPositive rate:", df_igg['label'].mean())

Raw shape: (9824, 20)
Long shape: (137536, 10)
Format counts:
format
IgG    68768
Fab    68768
Name: count, dtype: int64

IgG shape: 68768
After dropna: 57124
Rows per antibody:
antibody
1A2            7365
1B6            7385
Clesrovimab    8639
Nirsevimab     8402
Palivizumab    8396
RSM01          8675
Suptavumab     8262
dtype: int64

Positive rate: 0.04031580421539108


In [8]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Reload
df_igg = pd.read_csv("../data/RSV_F/rsv_igg_features.csv")
df_igg['label'] = (df_igg['escape_mean'] > 0.5).astype(int)
df_igg['seq_idx'] = df_igg['sequential_site'] - 1
y = df_igg['label'].values
abs_ = sorted(df_igg['antibody'].unique())
print(f"Rows: {len(df_igg)}, Antibodies: {len(abs_)}, PR: {y.mean():.4f}")

# Features
df_igg['wt_mut'] = df_igg['wildtype'] + df_igg['mutant']
enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.float32)
X_cat = enc.fit_transform(df_igg[['wt_mut']]).astype(np.float32)
X_pos = df_igg[['seq_idx']].values.astype(np.float32)
rsv_emb = np.load("../data/RSV_F/esm_cache/RSV_F_esm_35M.npy")
X_g = rsv_emb[df_igg['seq_idx'].values].astype(np.float32)

def xgb():
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
        eval_metric='logloss', random_state=42, n_jobs=2, tree_method='hist',
        max_bin=128, subsample=0.8, colsample_bytree=0.5)

def loao(X, y):
    out = {}
    for a in abs_:
        te = (df_igg['antibody'] == a).values; tr = ~te
        if len(np.unique(y[te])) < 2: continue
        s = xgb().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
        out[a] = roc_auc_score(y[te], s)
    return np.mean(list(out.values()))

def persite(y):
    au = []
    for a in abs_:
        te = (df_igg['antibody'] == a).values; tr = ~te
        m = df_igg[tr].assign(l=y[tr]).groupby('seq_idx')['l'].mean()
        s = df_igg.loc[te,'seq_idx'].map(m).fillna(y[tr].mean()).values
        au.append(roc_auc_score(y[te], s))
    return np.mean(au)

# Run at three thresholds
for name, thr in [("0.5", 0.5),
                  ("top10pct", df_igg['escape_mean'].quantile(0.9)),
                  ("top5pct",  df_igg['escape_mean'].quantile(0.95))]:
    y = (df_igg['escape_mean'] > thr).astype(int).values
    print(f"\n=== {name} (thr={thr:.4f}, PR={y.mean():.4f}) ===")
    a_ = loao(np.hstack([X_cat, X_pos]), y)
    b_ = loao(np.hstack([X_cat, X_pos, X_g]), y)
    p_ = persite(y)
    shuf = rsv_emb[np.random.RandomState(0).permutation(len(rsv_emb))][df_igg['seq_idx'].values]
    e_ = loao(np.hstack([X_cat, X_pos, shuf]), y)
    rnd = np.random.RandomState(1).randn(*rsv_emb.shape).astype(np.float32)[df_igg['seq_idx'].values]
    f_ = loao(np.hstack([X_cat, X_pos, rnd]), y)
    print(f"  A={a_:.4f}  B={b_:.4f}  per-site={p_:.4f}  shuffled={e_:.4f}  random={f_:.4f}")
    print(f"  B-A={b_-a_:+.4f}  B-per-site={b_-p_:+.4f}  B-shuf={b_-e_:+.4f}  B-rand={b_-f_:+.4f}")

Rows: 68768, Antibodies: 7, PR: 0.0335

=== 0.5 (thr=0.5000, PR=0.0335) ===
  A=0.6022  B=0.5992  per-site=0.5765  shuffled=0.5988  random=0.5951
  B-A=-0.0030  B-per-site=+0.0227  B-shuf=+0.0004  B-rand=+0.0041

=== top10pct (thr=0.2648, PR=0.0831) ===
  A=0.5895  B=0.6005  per-site=0.5708  shuffled=0.6010  random=0.6008
  B-A=+0.0110  B-per-site=+0.0298  B-shuf=-0.0005  B-rand=-0.0003

=== top5pct (thr=0.4395, PR=0.0415) ===
  A=0.5966  B=0.5984  per-site=0.5762  shuffled=0.6028  random=0.6014
  B-A=+0.0018  B-per-site=+0.0222  B-shuf=-0.0044  B-rand=-0.0030
